# Lab 3: Diagnosing Image Failures

**Workshop 2 · Block 2 · 25 minutes**

Level 3 is 40% of the preliminary round, and most teams score zero on it after the homework. That is the expected starting point.

When the answer is only inside a picture, no amount of chunking or prompt tuning reaches it. "My bot cannot answer image questions" has three different causes, and each needs a different fix:

| Cause | What happened | Fixed by |
|---|---|---|
| **A** Never collected | The scraper ignored the `<img>` tags | Collecting images while scraping (Step 2) |
| **B** Never described | You have the image URL, but nothing turned the picture into text | Describing images at build time (lab 4) |
| **C** Described badly | There is a description, but it is a caption, not the facts | A better description prompt (lab 4) |

**What you will do**

1. Find the image questions your bot fails
2. Check cause A: were the images collected?
3. Check cause B: were they described?
4. Check cause C: is a lazy description good enough?
5. Write down your diagnosis, A, B or C, for each question

> **You need:** the `images/` folder (in the workshop folder) and `dev_set.json`. The index from lab 2 (`data/chroma`) is used if you have it; if not, Step 1 builds one for you.
>
> **You leave with:** `data/images.json`, a list of every image found, and an A/B/C diagnosis for each image question.

> **How to use this notebook**
>
> 1. Run the cells **in order, top to bottom** (Shift+Enter).
>    `# ▶ RUN` cells just run. `# ✏️ EDIT` cells need you to change something first.
> 2. Most steps end with a **✅ Check** cell. If it prints ❌, fix that before moving on. The message says how.
> 3. If the kernel restarts, run the three **Setup** cells again, then carry on from where you were.
> 4. **Turn your VPN off.** The gateway rejects VPN addresses with a 403.

---
## Setup (2 minutes)

Three cells. Run all three before anything else. They are the same in labs 3, 4 and 5, so each notebook works on its own.

Your key is asked for with `getpass`: it is not shown, not saved, and gone when the kernel stops. If you have a `.env` file with `AZURE_OPENAI_KEY` in it, you will not be asked. **Never commit a notebook with a key visible in its output.**

In [ ]:
# ▶ RUN  Setup 1 of 3: install the packages into THIS notebook's Python.
# Quick if they are already there. If it installs anything new, restart the
# kernel (Kernel > Restart) and carry on from Setup 2.
%pip install -q -r requirements.txt

In [ ]:
# ▶ RUN  Setup 2 of 3: connect to the gateway and test all three routes.
import getpass, json, os, re, time, statistics
from pathlib import Path
from openai import AzureOpenAI

try:                                  # optional .env, see .env.example
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2025-01-01-preview")
CHAT_BASE   = os.getenv("AZURE_CHAT_BASE",  "https://api-iw.azure-api.net/sig-shared-jpeast-increased")
EMBED_BASE  = os.getenv("AZURE_EMBED_BASE", "https://api-iw.azure-api.net/sig-embedding")

CHAT_DEPLOYMENT   = os.getenv("CHAT_DEPLOYMENT",   "gpt-4o-mini")
VISION_DEPLOYMENT = os.getenv("VISION_DEPLOYMENT", "gpt-5-mini")
EMBED_DEPLOYMENT  = os.getenv("EMBED_DEPLOYMENT",  "text-embedding-3-small")

# The gateway takes the FULL path as the endpoint, so each client is bound
# to one deployment. The chat route has no /openai segment and the
# embedding route does. That is correct: do not make them match.
CHAT_URL   = f"{CHAT_BASE}/deployments/{CHAT_DEPLOYMENT}/chat/completions?api-version={API_VERSION}"
VISION_URL = f"{CHAT_BASE}/deployments/{VISION_DEPLOYMENT}/chat/completions?api-version={API_VERSION}"
EMBED_URL  = f"{EMBED_BASE}/openai/deployments/{EMBED_DEPLOYMENT}/embeddings?api-version={API_VERSION}"

# .strip() matters: a pasted key often carries a trailing newline -> 401.
KEY = (os.getenv("AZURE_OPENAI_KEY") or getpass.getpass("Azure OpenAI key: ")).strip()


def _client(url):
    return AzureOpenAI(azure_endpoint=url, api_key=KEY, api_version=API_VERSION)


chat_client   = _client(CHAT_URL)
vision_client = _client(VISION_URL)
embed_client  = _client(EMBED_URL)

failed = []
for label, fn in [
    ("chat  ", lambda: chat_client.chat.completions.create(
        model=CHAT_DEPLOYMENT, max_tokens=5,
        messages=[{"role": "user", "content": "Reply with one word: connected"}]
     ).choices[0].message.content.strip()),
    ("vision", lambda: vision_client.chat.completions.create(
        model=VISION_DEPLOYMENT,
        messages=[{"role": "user", "content": "Reply with one word: connected"}]
     ).choices[0].message.content.strip()),
    ("embed ", lambda: f"{len(embed_client.embeddings.create(model=EMBED_DEPLOYMENT, input=['test']).data[0].embedding)} dimensions"),
]:
    try:
        print(f"{label}  OK      {fn()}")
    except Exception as e:
        failed.append(str(e))
        print(f"{label}  FAILED  {type(e).__name__}: {str(e)[:110]}")

if not failed:
    print("\n✅ Connected. Carry on.")
elif any("403" in f for f in failed):
    print("\n❌ 403 Forbidden: your VPN is on. Turn it off and run this cell again.")
elif any("401" in f for f in failed):
    print("\n❌ 401: the key is wrong. Run this cell again and paste it carefully.")
else:
    print("\n❌ Not connected. 404 means the URL is wrong (did you edit this cell?). Ask a mentor.")

In [ ]:
# ▶ RUN  Setup 3 of 3: helper functions. Nothing to change here.

def chat(messages, model=None, temperature=0.0, max_tokens=512):
    r = chat_client.chat.completions.create(
        model=model or CHAT_DEPLOYMENT, messages=messages,
        temperature=temperature, max_tokens=max_tokens)
    return r.choices[0].message.content or ""


def ask(prompt, system=None, **kw):
    msgs = ([{"role": "system", "content": system}] if system else [])
    return chat(msgs + [{"role": "user", "content": prompt}], **kw)


def embed(texts, batch_size=256):
    """Embed a LIST of strings in batches (one call per string is slow)."""
    texts = [t.replace("\n", " ") for t in texts]
    out = []
    for i in range(0, len(texts), batch_size):
        r = embed_client.embeddings.create(model=EMBED_DEPLOYMENT, input=texts[i:i + batch_size])
        out.extend(d.embedding for d in r.data)
    return out


def chunk(text, size=800, overlap=100):
    text = " ".join(text.split())
    step = size - overlap
    return [text[i:i + size] for i in range(0, len(text), step) if text[i:i + size].strip()]


# One PersistentClient per folder for the life of the kernel. Opening a
# second one on the same folder corrupts the connection ("attempt to write
# a readonly database"), so rebuilds reset the COLLECTION, not the folder.
_stores = {}


def get_store(path="data/chroma", name="workshop", reset=False):
    import chromadb, shutil
    if path not in _stores:
        if reset and Path(path).exists():
            shutil.rmtree(path)
        try:
            _stores[path] = chromadb.PersistentClient(path=path)
        except KeyError as e:
            raise RuntimeError(
                f"chromadb cannot read the index at {path} ({e}). It was built by a "
                f"different chromadb version. Delete that folder and run again.") from None
    client = _stores[path]
    if reset:
        try:
            client.delete_collection(name)
        except Exception:
            pass
    return client.get_or_create_collection(name)


def ensure_index(path="data/chroma", name="workshop"):
    """Open the index lab 2 built. If it is missing or empty, build it from
    sample_corpus/ so this notebook still works if you skipped lab 2."""
    store = get_store(path, name)
    if store.count() == 0:
        print("No lab 2 index found, so building one from sample_corpus/ (about 30 seconds)...")
        texts, metas = [], []
        for f in sorted(Path("sample_corpus").glob("*.txt")):
            for i, piece in enumerate(chunk(f.read_text(encoding="utf-8"))):
                texts.append(piece)
                metas.append({"url": f.name, "position": i, "kind": "text"})
        store = get_store(path, name, reset=True)
        add_to_store(store, texts, metas)
    return store


def add_to_store(store, texts, metadatas, ids=None, batch_size=128):
    ids = ids or [f"c{i}" for i in range(len(texts))]
    for i in range(0, len(texts), batch_size):
        sl = slice(i, i + batch_size)
        store.add(ids=ids[sl], documents=texts[sl],
                  embeddings=embed(texts[sl]), metadatas=metadatas[sl])


def query(store, question, k=5, where=None):
    """The k nearest chunks, closest first. Distance is squared L2: lower is closer.
    `where` filters on metadata first. One field: {"kind": "image"}.
    Two or more fields need $and: {"$and": [{"year": "2025"}, {"type": "poster"}]}."""
    r = store.query(query_embeddings=embed([question]), n_results=k, where=where or None)
    return [{"text": d, "metadata": m, "distance": dist}
            for d, m, dist in zip(r["documents"][0], r["metadatas"][0], r["distances"][0])]


def show(chunks, chars=200):
    if not chunks:
        print("  (nothing returned)")
        return
    for c in chunks:
        print(f"  {c['distance']:.3f}  {c['metadata'].get('url', '?')}")
        print(f"         {c['text'][:chars].strip()}\n")


def timed(fn, *a, **kw):
    t0 = time.time()
    return fn(*a, **kw), time.time() - t0


def normalise(s):
    return re.sub(r"[^a-z0-9 ]", " ", (s or "").lower())


def answer_present(expected, chunks):
    """Is the expected answer anywhere in the retrieved text? A crude string
    match: read the chunks before you believe a FOUND."""
    hay  = normalise(" ".join(c["text"] for c in chunks))
    need = normalise(expected).strip()
    if need and need in hay:
        return True
    terms = [t for t in need.split() if len(t) > 3]
    return bool(terms) and sum(t in hay for t in terms) / len(terms) >= 0.8


def load_dev_set(path="dev_set.json"):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def describe_image(image_path, prompt, model=None):
    """One image plus an instruction, sent to the vision deployment.
    There is deliberately no default prompt: writing it is lab 4."""
    import base64, mimetypes
    p    = Path(image_path)
    mime = mimetypes.guess_type(p.name)[0] or "image/jpeg"
    b64  = base64.b64encode(p.read_bytes()).decode()
    r = vision_client.chat.completions.create(
        model=model or VISION_DEPLOYMENT,
        messages=[{"role": "user", "content": [
            {"type": "text", "text": prompt},
            {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{b64}"}},
        ]}])
    return r.choices[0].message.content or ""


# Where each workshop photo was taken. Lab 4 stores this as metadata and
# lab 5 filters on it.
PHOTO_ZONES = {
    "Brainstorming aerial.JPG": "Brainstorming Area",
    "Makerspace close up.JPG":  "Makerspace A",
    "makerspace.JPG":           "Makerspace A",
    "Open Event .JPG":          "Open Event Area",
    "Open event.JPG":           "Open Event Area",
}

print("✅ helpers loaded")

---
## Step 1 of 5 · Which image questions do you fail? (4 minutes)

**Goal:** ask your text-only index the two dev-set questions whose answer is in a picture, and see whether the answer appears in the five chunks that come back.

**How to read the result**

- **`MISSING`**: the answer is not in any text your bot can retrieve, so no prompt can produce it. Steps 2 to 4 find out why.
- **`FOUND`**: the answer's words appear somewhere in the retrieved text. It is a crude string match, so **read the chunks before you believe it.** The Lv5 answer, "Innovation", is part of the Centre's name on almost every page, so it says FOUND even though nothing describes the wall.
- **`retrieved`**: the source page of each chunk and its distance (lower is closer). The index finds pages about the right room or event, but those pages do not contain the fact.

In [ ]:
# ▶ RUN
store = ensure_index()                      # the lab 2 index, or a fresh one
dev   = load_dev_set("dev_set.json")
image_qs = [d for d in dev if d["level"] in (3, 5)]   # answers live in a picture

print(f"{store.count()} chunks in the index\n")
for n, item in enumerate(image_qs):
    hits    = query(store, item["question"], k=5)
    verdict = "FOUND" if answer_present(item["answer"], hits) else "MISSING"
    print(f"[{n}] Lv{item['level']}  {item['question']}")
    print(f"     expected: {item['answer']}")
    print(f"     answer in top 5 chunks: {verdict}")
    print("     retrieved:", ", ".join(f"{c['metadata'].get('url', '?')} ({c['distance']:.2f})" for c in hits))
    print()

**You should see:** the Lv3 question `MISSING`, and the Lv5 question `FOUND`. Now check that FOUND: run the next cell and read the Lv5 chunks. Does any of them describe the wall, or is "Innovation" only there as part of a name?

In [ ]:
# ✏️ EDIT the number (0 = the Lv3 question, 1 = the Lv5 question), then run
item = image_qs[1]
print("Q:", item["question"], "\n")
show(query(store, item["question"], k=5), chars=600)

---
## Step 2 of 5 · Cause A: was the image ever collected? (4 minutes)

If your scraper kept only text, this is the problem, and nothing further down matters yet.

The next cell reads one real Inno Wing page and records, for every `<img>` tag, four pieces of text. **It does not download any images.**

| Field | What it is |
|---|---|
| `src` | the image URL, made absolute with `urljoin` (relative links will not load later) |
| `alt` | the alt text, if the page has any |
| `caption` | the `<figcaption>`, if the image sits in a `<figure>` |
| `page` | the page it was found on, so you can trace a wrong answer back to it |

This is the same code as `extract()` in `submission_repo/build/scrape.py`. In your repository it runs over every page your scraper visits, in the same pass as the text; going back for images later means crawling both sites again.

In [ ]:
# ▶ RUN  (needs internet; takes a few seconds)
from urllib.parse import urljoin
from bs4 import BeautifulSoup
import requests

def extract_images(html, page_url):
    soup, out = BeautifulSoup(html, "html.parser"), []
    for img in soup.select("img"):
        src = img.get("src")
        if not src:
            continue
        fig = img.find_parent("figure")
        cap = fig.find("figcaption").get_text(strip=True) if fig and fig.find("figcaption") else ""
        out.append({"src": urljoin(page_url, src), "alt": img.get("alt", ""),
                    "caption": cap, "page": page_url})
    return out

PAGES = ["https://innowings.engg.hku.hk/innowing1/"]   # one real page as a demo

images = []
for url in PAGES:
    try:
        images += extract_images(requests.get(url, timeout=20).text, url)
    except Exception as exc:
        print("could not fetch", url, "->", exc)

Path("data").mkdir(exist_ok=True)
Path("data/images.json").write_text(json.dumps(images, indent=1), encoding="utf-8")
print(f"{len(images)} images -> data/images.json\n")
for im in images:
    print(f"  alt {im['alt'] or '(none)':<18}  {im['src'].rsplit('/', 1)[-1][:55]}")

In [ ]:
# ✅ Check
recs = json.loads(Path("data/images.json").read_text(encoding="utf-8"))
no_alt = sum(1 for r in recs if not r["alt"].strip())
if not recs:
    print("❌ No images collected. The page could not be fetched: check your internet and run the cell above again.")
else:
    print(f"✅ {len(recs)} images collected from one page. {no_alt} of them have no alt text,")
    print("   so the only way to their content is a vision model. A text-only scraper had none of them.")

---
## Step 3 of 5 · Cause B: was it ever described? (2 minutes)

An image URL in your data is not searchable. Something has to turn the picture into words before it can be embedded and matched. That happens **at build time, never while answering**: building has no time limit, answering has 30 seconds.

This cell counts what kinds of chunk your index holds.

In [ ]:
# ▶ RUN
metas = store.get(include=["metadatas"])["metadatas"]
kinds = {}
for m in metas:
    kinds[m.get("kind", "text")] = kinds.get(m.get("kind", "text"), 0) + 1
print("chunk kinds in your index:", kinds)

if "image" not in kinds:
    print("\nCause B: there are no image descriptions in the index. Lab 4 fixes this.")
else:
    print(f"\nYou already have {kinds['image']} image chunks (from lab 4?). Cause B is fixed for those.")

---
## Step 4 of 5 · Cause C: was it described badly? (6 minutes)

Now describe one photo with the laziest possible prompt, `"Describe this image."`, which is what most teams tried for the homework. The photo is shown above the description so you can compare.

The call takes **10 to 30 seconds**. Wait for it.

In [ ]:
# ▶ RUN
from IPython.display import Image as ShowImage, display

IMAGES = sorted(str(p) for p in Path("images").iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
assert IMAGES, "No photos in images/. It ships with the workshop folder: download it again."

photo = next((p for p in IMAGES if "brainstorming" in p.lower()), IMAGES[0])
display(ShowImage(filename=photo, width=600))

naive = describe_image(photo, "Describe this image.")
print(naive)

Look at the photo, then at the text. Fill in the next cell.

In [ ]:
# ✏️ EDIT, then run
missing_facts = [
    "",   # a fact you can SEE in the photo that the description left out
    "",   # another one
]
lv5_answerable = None   # True or False: could you answer the Lv5 wall question from this text alone?

In [ ]:
# ✅ Check
filled = [f for f in missing_facts if f.strip()]
if len(filled) < 2 or lv5_answerable is None:
    print("❌ Fill in both facts and lv5_answerable in the cell above, then run it and this cell again.")
else:
    print("✅ Good. Your two missing facts:")
    for f in filled:
        print("   -", f)
    print("\nLab 4 tests exactly this kind of fact: how many cushions are on the low bench,")
    print("and what is directly to the right of the tiered seating. A caption rarely has either.")
    if lv5_answerable:
        print("\nThe big wall sign often survives a lazy prompt. Counts and small print usually do not.")

---
## Step 5 of 5 · Write down your diagnosis (3 minutes)

The two image questions in `dev_set.json` fail for **different** causes. Hint: one answer is on a poster that exists only on the website. The other is on a wall in one of the five photos in `images/`.

Fix in the order A, then B, then C. A better description is no use for an image you never collected.

In [ ]:
# ✏️ EDIT: put "A", "B" or "C" for each, then run
DIAGNOSIS = {
    "Lv3 pitch winners": "?",
    "Lv5 wall words":    "?",
}

In [ ]:
# ✅ Check
EXPECTED = {
    "Lv3 pitch winners": ("A", "The winners are on a poster that only exists on the website, and the demo page in Step 2 "
                               "did not include it. It was never collected. Once collected it is also cause B until described."),
    "Lv5 wall words":    ("B", "The photo is in images/, but nothing turned it into text in the index. Once described, "
                               "whether the exact words survive depends on your prompt: that is cause C, and lab 4."),
}
for q, (want, why) in EXPECTED.items():
    got = DIAGNOSIS.get(q, "?").strip().upper()
    mark = "✅" if got == want else ("🤔" if got in ("A", "B", "C") else "❌")
    print(f"{mark} {q}: you said {got}, the main cause is {want}.")
    print(f"   {why}\n")

---
## What to take to your own repository

| Cause | Where you fix it |
|---|---|
| A | `build/scrape.py`: collect `<img>` records in the same pass as the text (Step 2 code) |
| B | `build/images.py`: describe every collected image once, at build time, and add the text to the index |
| C | `DESCRIPTION_PROMPT` in `build/images.py`: this is **lab 4** |

**Next: lab 4.**